# PopQA · GenRead: GPT-Neo 1.3B
Issue #27 · Notebook độc lập, dành cho Kaggle. **Chưa chạy model thật trên GPU.**

## Cách chạy phiên đầu
1. Import notebook này vào Kaggle, bật **GPU** và **Internet**. Không cần dataset BM25.
2. Giữ cấu hình mặc định: kiểm 300 câu cho GPT-Neo 1.3B, sau đó chạy full lần lượt GPT-Neo 1.3B.
3. **Save Version → Save & Run All**. Có thể tắt máy. Xem Output để lấy ZIP khi phiên kết thúc.
4. Nếu chỉ muốn kiểm thử: đặt `RUN_FULL = False` ở ô cấu hình.

Một GPU chạy GPT-Neo 1.3B; dùng được T4 ×2 hoặc P100 nhưng mặc định chỉ dùng GPU 0.
Mặc định dừng ở mốc 11 giờ tính từ cell đầu, chừa thời gian lưu output. Đây là ngân sách do người chạy đặt,
không phải cam kết về giới hạn phiên/quota Kaggle. Hãy giảm nếu tài khoản còn ít giờ.

## Chạy tiếp phiên sau
Add Input output phiên trước có **popqa-genread-neo13b-checkpoint.zip**, giữ nguyên notebook và Run All.
Notebook tự tìm ZIP, kiểm cấu hình, bỏ qua CSV đã hoàn thành, tiếp tục cả pilot/full đang dở.
Nếu nhiều ZIP, đặt `RESTORE_FROM` là đường dẫn ZIP muốn dùng. Không dùng checkpoint vanilla/BM25.
Không nối 300 câu vào full: mỗi lượt có kế hoạch chọn few-shot riêng theo code gốc.

## Thiết lập và khác biệt cần ghi trong báo cáo
- fp32; seed 633; 15 ví dụ thuộc 15 quan hệ khác; greedy; ngữ cảnh 2048 token cắt trái.
- Sinh tối đa **150 token** đoạn nền, rồi **15 token** trả lời. Giữ cách lấy **dòng đầu** của code gốc cho cả hai bước.
- GenRead bốc một ví dụ cho mỗi trong 16 quan hệ một lần mỗi lượt; mỗi câu dùng 15 ví dụ khác quan hệ.
- **Sửa lookup few-shot** để ví dụ có đoạn GenRead đã sinh, tương tự cách nhóm dùng BM25 có đoạn văn.
  Đây không phải chạy nguyên xi bug của upstream. Đoạn sinh cho ví dụ và câu chính đều được lưu.
- `is_correct` giữ substring metric gốc; `has_answer` chỉ xét `gen_passage`, không xét prompt/đáp án mẫu.
- `conf` là trung bình log-xác suất token câu trả lời; để trống khi không có token trả lời hợp lệ.
- Pin revision model, phiên bản transformers, hash dữ liệu và mã runner. Đổi cấu hình ảnh hưởng kết quả
  thì checkpoint bị từ chối. Đổi ngân sách giờ hoặc `RUN_FULL` không làm mất checkpoint.
- Kết quả thật chỉ có sau khi bạn chạy Kaggle; không có accuracy dự kiến được điền sẵn.

Tham khảo tại commit `2706fc6` của [CS221](https://github.com/vanductan-NLT/CS221):
[OPT notebook](https://github.com/vanductan-NLT/CS221/blob/2706fc63e34b741575706a6f50fd9d22f9ac60fe/notebooks/opt_1_3b_2_7b_phien1.ipynb),
[checkpoint notebook](https://github.com/vanductan-NLT/CS221/blob/2706fc63e34b741575706a6f50fd9d22f9ac60fe/notebooks/opt_6_7b_fp16.ipynb),
[GenRead gốc](https://github.com/vanductan-NLT/CS221/blob/2706fc63e34b741575706a6f50fd9d22f9ac60fe/upstream/run_model.py).

In [1]:
import time, sys, subprocess
SESSION_START = time.time()
TIME_BUDGET_HOURS = 11.0
RUN_FULL = True
RESTORE_FROM = 'auto'  # hoặc đường dẫn ZIP chính xác; None = không import checkpoint
POLL_SECONDS = 30
BACKUP_EVERY_SECONDS = 600
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'transformers==4.57.6', 'pandas', 'numpy', 'huggingface_hub'])

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


0

## 1. Chuẩn bị dữ liệu và khôi phục phiên trước
PopQA và 300 ID được kiểm hash/thứ tự. ZIP không được phép ghi file ngoài thư mục làm việc.

In [2]:
from pathlib import Path
import os, json, hashlib, zipfile, platform, importlib.metadata, urllib.request
import pandas as pd
import numpy as np
import torch
from IPython.display import display, FileLink
assert torch.cuda.is_available(), 'Bật GPU trong Kaggle Settings.'
print('GPU:', torch.cuda.get_device_name(0))
work = Path('/kaggle/working/popqa-genread-neo13b')
work.mkdir(parents=True, exist_ok=True)
for name in ['data', 'results', 'checkpoints', 'logs']:
    (work / name).mkdir(exist_ok=True)
if RESTORE_FROM == 'auto':
    found = sorted(Path('/kaggle/input').rglob('popqa-genread-neo13b-checkpoint.zip'))
    assert len(found) <= 1, 'Nhiều ZIP checkpoint: điền RESTORE_FROM cụ thể.'
    RESTORE_FROM = found[0] if found else None
if RESTORE_FROM:
    with zipfile.ZipFile(RESTORE_FROM) as z:
        assert z.testzip() is None, 'ZIP bị hỏng.'
        for info in z.infolist():
            if info.is_dir():
                continue
            rel = Path(info.filename)
            assert not rel.is_absolute() and '..' not in rel.parts
            assert rel.parts[0] in {'checkpoints', 'results', 'logs', 'run_config.json'}
            target = work / rel
            content = z.read(info)
            if target.exists():
                assert target.read_bytes() == content, f'File hiện tại khác ZIP: {rel}. Dùng phiên Kaggle mới.'
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(content)
    print('Đã khôi phục:', RESTORE_FROM)
else:
    print('Không import ZIP; giữ checkpoint trong working nếu đang chạy lại cùng phiên.')
REPO_COMMIT = '2706fc63e34b741575706a6f50fd9d22f9ac60fe'
data_path = work / 'data/popQA.tsv'
if not data_path.exists():
    urllib.request.urlretrieve(f'https://raw.githubusercontent.com/vanductan-NLT/CS221/{REPO_COMMIT}/upstream/data/popQA.tsv', data_path)
assert hashlib.sha256(data_path.read_bytes().replace(b'\r\n', b'\n')).hexdigest() == '9a5227f41bff0e4c331d4a774d946b12f95307892b58f860a9606ef356e6089b'
questions = pd.read_csv(data_path, sep='\t')
assert len(questions) == 14267 and questions.id.is_unique
WANT_IDS = [5139062, 4638370, 1353394, 5923312, 3356649, 4996780, 743705, 2357044, 4066277, 1642428, 4188191, 605897, 1473780, 1223424, 5999251, 5948044, 780737, 6378269, 994647, 4409246, 4522741, 837389, 2671483, 3804084, 1872067, 666610, 656905, 2250382, 1791448, 2485455, 6506411, 5179731, 5789578, 6365456, 5422818, 2174102, 5006891, 5781178, 2990323, 184846, 2556139, 5557556, 4330249, 2507186, 946336, 2691112, 2226731, 3974408, 2449796, 4921831, 1646720, 1342774, 1481181, 1977383, 3793259, 192261, 2987359, 237759, 264374, 3720231, 1237178, 2647530, 4234132, 2071152, 1306567, 1228958, 1594184, 3826516, 1194410, 5960486, 5560949, 4496891, 1387057, 232809, 4997435, 388189, 5580295, 5982132, 2105565, 1969958, 3106384, 246959, 2763878, 5303917, 3868066, 2281877, 4393083, 2899611, 1574139, 4613808, 1804783, 1187163, 3467441, 788195, 5839103, 3672517, 2068848, 2194088, 1860470, 1001346, 1225029, 776128, 2654404, 1391948, 3315030, 1116447, 3784290, 4331418, 1054401, 5187250, 2849306, 3755811, 832908, 1549444, 1616396, 1358512, 447830, 2254982, 5918313, 4666651, 596547, 1953459, 2647017, 1680080, 4684145, 3243026, 1316602, 5929687, 1001108, 935146, 2908490, 5920784, 2592338, 1486783, 6281394, 4270058, 2073781, 6004711, 5918653, 3956481, 2833871, 2435793, 5945781, 3423081, 4404592, 6547647, 5846736, 5937854, 3950149, 4763253, 3633815, 288745, 2059485, 4652935, 1932836, 2630371, 2213921, 977275, 2766419, 636616, 144024, 2085939, 1447840, 1758617, 3524273, 5915241, 37343, 3677957, 2963470, 3080610, 5773632, 2839162, 1660687, 6502612, 1332334, 5962937, 1571598, 3087676, 5669872, 4265255, 2229505, 5137613, 1913397, 5592430, 1074278, 3763572, 196395, 3276946, 5113116, 2374633, 1919697, 1107870, 5045808, 6374945, 2663309, 2311628, 5088311, 6042636, 614930, 432839, 3520627, 1402351, 843797, 75263, 35, 244122, 5777706, 474017, 6532951, 4532127, 3095296, 2886182, 1570230, 3675616, 4310070, 701352, 3450665, 186809, 3903448, 3952467, 5041070, 3514599, 6081120, 3543733, 3294045, 5547893, 4329831, 6533919, 5263647, 4801270, 6385442, 1162864, 4484472, 215156, 1504171, 5951099, 2223262, 5644258, 1855691, 1035763, 2756925, 1466602, 1868517, 6161854, 3422786, 4584501, 380223, 3981349, 2664757, 5485882, 3178175, 4405633, 1401285, 1349825, 3183040, 790430, 3845086, 1120961, 1134562, 285213, 632979, 353985, 285832, 1997293, 201733, 72369, 2897641, 595358, 4968395, 1001697, 1108230, 902091, 5970015, 144894, 398909, 2951170, 3172905, 3261768, 5169150, 500365, 5777300, 2114057, 1117503, 617409, 4365138, 394543, 691428, 1722398, 5770014, 6521985, 1081675, 2181196, 5979071, 4557797, 2961443, 1717233, 294568, 6001463, 2379460, 2245901]
assert questions.sample(n=300, random_state=633).id.astype(int).tolist() == WANT_IDS
print('PASS: PopQA 14.267 câu và 300 ID cố định.')

GPU: Tesla T4
Không import ZIP; giữ checkpoint trong working nếu đang chạy lại cùng phiên.
PASS: PopQA 14.267 câu và 300 ID cố định.


## 2. Runner GenRead và cấu hình tái lập
Runner được nhúng ngay trong notebook; không cần upload file Python riêng.

In [3]:
WORKER_SOURCE = '"""GenRead runner embedded in the standalone Kaggle notebook. No GPU on import."""\nimport argparse\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport random\nimport time\n\nimport numpy as np\nimport pandas as pd\n\nPIDS = [22, 218, 91, 257, 182, 164, 526, 97, 533, 639, 472, 106, 560, 484, 292, 422]\nGEN_TEMPLATE = \'Generate a background document from Wikipedia to answer the given question. {}\'\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(path.suffix + \'.tmp\')\n    with temp.open(\'w\', encoding=\'utf-8\') as f:\n        json.dump(value, f, ensure_ascii=False, allow_nan=False)\n        f.flush()\n        os.fsync(f.fileno())\n    os.replace(temp, path)\n\n\ndef append_record(path, record):\n    with Path(path).open(\'ab\') as f:\n        f.write((json.dumps(record, ensure_ascii=False, allow_nan=False) + \'\\n\').encode())\n        f.flush()\n        os.fsync(f.fileno())\n\n\ndef read_journal(path, header):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    if not path.exists():\n        append_record(path, header)\n    records = []\n    with path.open(\'rb+\') as f:\n        first = f.readline()\n        if not first.endswith(b\'\\n\') or json.loads(first) != header:\n            raise ValueError(\'Checkpoint khác cấu hình/model/dữ liệu. Không ghi đè.\')\n        while True:\n            offset = f.tell()\n            line = f.readline()\n            if not line:\n                break\n            if not line.endswith(b\'\\n\'):\n                f.seek(offset)\n                f.truncate()\n                f.flush()\n                os.fsync(f.fileno())\n                break\n            records.append(json.loads(line))\n    return records\n\n\ndef correct(answers, text):\n    # Preserve the original, deliberately permissive substring metric.\n    return any(a in text or a.lower() in text or a.capitalize() in text for a in answers)\n\n\ndef first_line(text):\n    return (text[2:] if text.startswith(\'\\n\\n\') else text).split(\'\\n\')[0]\n\n\ndef make_plan(questions, sample):\n    # Local RNGs reproduce upstream seed resets, independent of model loading.\n    rng = np.random.RandomState(633)\n    py_rng = random.Random(633)\n    selected = questions.sample(n=sample or len(questions), random_state=rng)\n    demos = {}\n    for pid in PIDS:\n        demos[pid] = questions[questions.prop_id == pid].sample(n=1, random_state=rng).iloc[0].to_dict()\n    plans = []\n    for row in selected.itertuples(index=False):\n        other = [p for p in PIDS if p != row.prop_id]\n        order = [py_rng.sample([p], 1)[0] for p in other]\n        rng.shuffle(order)\n        plans.append((int(row.id), order))\n    return selected, demos, plans\n\n\ndef prompt_for(row, order, demos, passages, passage):\n    shots = [passages[p] + \'\\n\\nQ: \' + demos[p][\'question\'] + \' A: \' + demos[p][\'obj\'] for p in order]\n    return \'\\n\\n\'.join(shots) + \'\\n\\n\' + passage + \'\\n\\nQ: \' + row[\'question\'] + \' A:\'\n\n\ndef evaluate(questions, sample, generate, root, identity, expired=lambda: False):\n    """Checkpoint every demo, generated question passage and completed answer."""\n    root = Path(root)\n    selected, demos, plans = make_plan(questions, sample)\n    ids = selected.id.astype(int).tolist()\n    header = dict(version=1, identity=identity, ids=ids)\n    answers = read_journal(root / \'answers.jsonl\', header)\n    for i, record in enumerate(answers):\n        if i >= len(ids) or record[\'id\'] != ids[i]:\n            raise ValueError(\'Checkpoint answers phải là prefix đúng thứ tự, không trùng ID.\')\n    passages_raw = read_journal(root / \'passages.jsonl\', header)\n    passages = {}\n    allowed = {f\'demo:{p}\' for p in PIDS} | {f\'question:{i}\' for i in ids}\n    for r in passages_raw:\n        if r[\'key\'] not in allowed or r[\'key\'] in passages:\n            raise ValueError(\'Passage cache có key không hợp lệ/trùng.\')\n        passages[r[\'key\']] = r\n    def passage(key, question):\n        text = GEN_TEMPLATE.format(question)\n        if key in passages:\n            if passages[key][\'prompt\'] != text:\n                raise ValueError(\'Prompt sinh passage khác checkpoint.\')\n            return passages[key][\'text\']\n        if expired():\n            raise TimeoutError(\'Đến giờ lưu checkpoint.\')\n        result = generate(text, 150)\n        r = dict(key=key, prompt=text, text=result[\'pred\'])\n        append_record(root / \'passages.jsonl\', r)\n        passages[key] = r\n        return r[\'text\']\n    demo_passages = {p: passage(f\'demo:{p}\', demos[p][\'question\']) for p in PIDS}\n    indexed = selected.set_index(\'id\', drop=False)\n    started, initial = time.monotonic(), len(answers)\n    for index, (qid, order) in enumerate(plans):\n        row = indexed.loc[qid].to_dict()\n        generated = passage(f\'question:{qid}\', row[\'question\'])\n        prompt = prompt_for(row, order, demos, demo_passages, generated)\n        labels = json.loads(row[\'possible_answers\'])\n        if index < len(answers):\n            record = answers[index]\n            if (record[\'prompt\'] != prompt or record[\'gen_passage\'] != generated\n                    or record[\'is_correct\'] != correct(labels, record[\'pred\'])\n                    or record[\'has_answer\'] != correct(labels, generated)):\n                raise ValueError(f\'Prompt/nhãn resume khác ở ID {qid}\')\n            continue\n        if expired():\n            raise TimeoutError(\'Đến giờ lưu checkpoint.\')\n        out = generate(prompt, 15)\n        record = dict(id=qid, prompt=prompt, pred=out[\'pred\'], generation=out[\'response\'],\n                      conf=out[\'conf\'], gen_passage=generated,\n                      is_correct=correct(labels, out[\'pred\']), has_answer=correct(labels, generated))\n        append_record(root / \'answers.jsonl\', record)\n        answers.append(record)\n        if len(answers) % 10 == 0 or len(answers) == len(ids):\n            elapsed = time.monotonic() - started\n            speed = elapsed / max(1, len(answers) - initial)\n            atomic_json(root / \'progress.json\', dict(done=len(answers), total=len(ids), seconds_per_new_answer=speed))\n            print(f\'{len(answers)}/{len(ids)} | {speed:.2f}s/câu mới\', flush=True)\n    frame = selected.reset_index(drop=True).copy()\n    for key in answers[0]:\n        if key != \'id\':\n            frame[key] = [r[key] for r in answers]\n    return frame\n\n\ndef verify(frame, questions, sample, root, identity):\n    selected, demos, plans = make_plan(questions, sample)\n    assert frame.id.astype(int).tolist() == selected.id.astype(int).tolist(), \'Sai ID hoặc thứ tự\'\n    assert frame.id.is_unique\n    header = dict(version=1, identity=identity, ids=selected.id.astype(int).tolist())\n    cached = read_journal(Path(root) / \'passages.jsonl\', header)\n    cache = {r[\'key\']: r for r in cached}\n    assert len(cache) == len(cached)\n    dp = {}\n    for p in PIDS:\n        r = cache[f\'demo:{p}\']\n        assert r[\'prompt\'] == GEN_TEMPLATE.format(demos[p][\'question\'])\n        dp[p] = r[\'text\']\n    for (_, order), row in zip(plans, frame.to_dict(\'records\')):\n        passage = cache[f\'question:{int(row["id"])}\']\n        assert passage[\'prompt\'] == GEN_TEMPLATE.format(row[\'question\'])\n        assert row[\'gen_passage\'] == passage[\'text\']\n        assert row[\'prompt\'] == prompt_for(row, order, demos, dp, passage[\'text\'])\n        labels = json.loads(row[\'possible_answers\'])\n        assert str(row[\'is_correct\']).lower() == str(correct(labels, row[\'pred\'])).lower()\n        assert str(row[\'has_answer\']).lower() == str(correct(labels, row[\'gen_passage\'])).lower()\n        if row[\'conf\'] not in (\'\', None):\n            assert math.isfinite(float(row[\'conf\'])) and float(row[\'conf\']) <= 1e-5\n    return True\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument(\'--model\', required=True)\n    p.add_argument(\'--revision\', required=True)\n    p.add_argument(\'--sample\', type=int, choices=[0, 300], required=True)\n    p.add_argument(\'--device\', default=\'cuda:0\')\n    p.add_argument(\'--deadline\', type=float, required=True)\n    p.add_argument(\'--config\', required=True)\n    args = p.parse_args()\n    config = json.loads(Path(args.config).read_text())\n    identity = dict(config=config, model=args.model, revision=args.revision, sample=args.sample)\n    phase = \'check300\' if args.sample else \'full\'\n    slug = args.model.replace(\'/\', \'_\')\n    root = Path(\'checkpoints\') / slug / phase\n    questions = pd.read_csv(\'data/popQA.tsv\', sep=\'\\t\')\n    selected, _, _ = make_plan(questions, args.sample)\n    n = len(selected)\n    dest = Path(\'results\') / f\'model={slug}-input=popqa-genread-shot-passages-{phase}-method=genread-shots=15-n={n}.csv\'\n    meta = dest.with_suffix(\'.run.json\')\n    if dest.exists():\n        assert json.loads(meta.read_text()) == identity, \'CSV khác cấu hình.\'\n        frame = pd.read_csv(dest, keep_default_na=False)\n        verify(frame, questions, args.sample, root, identity)\n        print(\'Đã có CSV đầy đủ và đã kiểm lại:\', dest, flush=True)\n        return\n    expired = lambda: time.time() >= args.deadline\n    if expired():\n        raise SystemExit(75)\n    import torch\n    from transformers import AutoTokenizer, AutoModelForCausalLM\n    random.seed(633)\n    np.random.seed(633)\n    torch.manual_seed(633)\n    torch.cuda.manual_seed_all(633)\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cuda.matmul.allow_tf32 = False\n    tokenizer = AutoTokenizer.from_pretrained(args.model, revision=args.revision)\n    model = AutoModelForCausalLM.from_pretrained(args.model, revision=args.revision,\n                torch_dtype=torch.float32, attn_implementation=\'eager\').eval().to(args.device)\n    print(\'MODEL READY\', args.model, next(model.parameters()).dtype, flush=True)\n    def generate(prompt, tokens):\n        inputs = tokenizer(prompt, return_tensors=\'pt\').to(args.device)\n        inputs = {k: v[:, -(2048 - tokens):] for k, v in inputs.items()}\n        with torch.inference_mode():\n            out = model.generate(**inputs, max_new_tokens=tokens, num_beams=1, do_sample=False,\n                                 pad_token_id=tokenizer.eos_token_id,\n                                 return_dict_in_generate=True, output_scores=(tokens == 15))\n        response = tokenizer.decode(out.sequences[0])\n        actual = tokenizer.decode(inputs[\'input_ids\'][0])\n        pred = first_line(response[len(actual):])\n        conf = None\n        if tokens == 15 and pred:\n            scores = model.compute_transition_scores(out.sequences, out.scores, normalize_logits=True)[0]\n            new_ids = out.sequences[0, inputs[\'input_ids\'].shape[1]:]\n            decode = lambda x: tokenizer.decode(x, clean_up_tokenization_spaces=False)\n            text = decode(new_ids)\n            start = 2 if text.startswith(\'\\n\\n\') else 0\n            end = text.find(\'\\n\', start)\n            end = len(text) if end < 0 else end\n            picked, prev = [], 0\n            for k in range(len(scores)):\n                cur = len(decode(new_ids[:k + 1]))\n                if cur > start and prev < end:\n                    picked.append(float(scores[k]))\n                prev = cur\n            conf = sum(picked) / len(picked) if picked else None\n        return dict(pred=pred, response=response, conf=conf)\n    try:\n        frame = evaluate(questions, args.sample, generate, root, identity, expired)\n    except TimeoutError:\n        print(\'TẠM DỪNG: checkpoint đã lưu.\', flush=True)\n        raise SystemExit(75)\n    verify(frame, questions, args.sample, root, identity)\n    dest.parent.mkdir(exist_ok=True)\n    atomic_json(meta, identity)\n    temp = dest.with_suffix(\'.csv.tmp\')\n    frame.to_csv(temp, index=False)\n    os.replace(temp, dest)\n    lengths = [len(tokenizer(text, truncation=False)[\'input_ids\']) for text in frame.prompt]\n    report = dict(model=args.model, method=\'genread\', n=n, accuracy=float(frame.is_correct.mean()),\n                  has_answer=float(frame.has_answer.mean()), empty_predictions=int(frame.pred.str.strip().eq(\'\').sum()),\n                  empty_passages=int(frame.gen_passage.str.strip().eq(\'\').sum()),\n                  truncated_prompts=sum(x > 2033 for x in lengths), max_prompt_tokens=max(lengths),\n                  ids_and_prompts_ok=True)\n    atomic_json(dest.with_suffix(\'.summary.json\'), report)\n    print(\'PASS\', json.dumps(report), flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n'
runner = work / "genread_worker.py"
runner.write_text(WORKER_SOURCE, encoding="utf-8")

12355

In [4]:
from huggingface_hub import model_info
MODELS = ['EleutherAI/gpt-neo-1.3B']
previous_path = work / 'run_config.json'
previous = json.loads(previous_path.read_text()) if previous_path.exists() else None
revisions = previous['model_revisions'] if previous else {m: model_info(m).sha for m in MODELS}
assert set(revisions) == set(MODELS)
config = dict(schema=1, method='genread', dtype='fp32', seed=633, shots=15,
              context=2048, answer_tokens=15, passage_tokens=150, shot_passages=True,
              passage_policy='upstream-first-line', has_answer_policy='generated-passage-only',
              attention='eager', model_revisions=revisions,
              worker_sha256=hashlib.sha256(runner.read_bytes()).hexdigest(),
              data_sha256=hashlib.sha256(data_path.read_bytes()).hexdigest(), subset_ids=WANT_IDS,
              python=platform.python_version(), gpu=torch.cuda.get_device_name(0),
              packages={p: importlib.metadata.version(p) for p in ['torch','transformers','pandas','numpy','huggingface_hub']})
if previous:
    assert previous == config, 'Môi trường/code/dữ liệu khác checkpoint; không trộn kết quả. Xem run_config.json.'
else:
    previous_path.write_text(json.dumps(config, indent=2), encoding='utf-8')
print('Model revisions:', revisions)
print('Run config SHA:', hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest())

Model revisions: {'EleutherAI/gpt-neo-1.3B': 'dbe59a7f4a88d01d1ba9798d78dbe3fe038792c8'}
Run config SHA: c2f231d4759877784178007ea6df0befcb6e3017db41a4bc1ef8ab90943a7e38


## 3. Điều khiển phiên và đóng gói tự động
Checkpoint append + flush/fsync sau mỗi đoạn sinh và câu trả lời. Nếu bị kill giữa một dòng cuối,
resume bỏ dòng dở rồi làm lại đúng bước đó; các dòng hoàn chỉnh phải có JSON hợp lệ.
ZIP checkpoint cập nhật mỗi 10 phút và trong `finally` khi dừng/lỗi/interrupt thông thường.
Nếu Kaggle kill cứng trước lúc lưu ZIP mới, ZIP có thể lùi tối đa một chu kỳ backup;
các file checkpoint thô trong output vẫn được giữ. Giữ ngân sách giờ có khoảng đệm.

In [5]:
def package_checkpoint():
    target = Path('/kaggle/working/popqa-genread-neo13b-checkpoint.zip')
    temp = target.with_suffix('.zip.tmp')
    files = [work / 'run_config.json']
    files += [p for folder in ['checkpoints','results','logs'] for p in (work / folder).rglob('*')
              if p.is_file() and not p.name.endswith('.tmp')]
    with zipfile.ZipFile(temp, 'w', zipfile.ZIP_DEFLATED) as z:
        for p in files:
            z.write(p, str(p.relative_to(work)))
    os.replace(temp, target)
    return target

def stop_process(proc):
    if proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=20)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait()

def run_phase(sample):
    phase = 'check300' if sample else 'full'
    deadline = SESSION_START + TIME_BUDGET_HOURS * 3600
    for model in MODELS:  # Chỉ GPT-Neo 1.3B; mỗi tiến trình giải phóng GPU khi thoát.
        if time.time() >= deadline:
            print('TẠM DỪNG: hết ngân sách phiên. Add Input checkpoint ZIP ở phiên sau.')
            return False
        command = [sys.executable, '-u', str(runner), '--model', model,
                   '--revision', revisions[model], '--sample', str(sample), '--device', 'cuda:0',
                   '--deadline', str(deadline), '--config', str(work / 'run_config.json')]
        log_path = work / 'logs' / (model.replace('/', '_') + '-' + phase + '.log')
        process = None
        last_backup = time.time()
        try:
            with log_path.open('a', encoding='utf-8') as log:
                process = subprocess.Popen(command, cwd=work, stdout=log, stderr=subprocess.STDOUT)
                print('Bắt đầu:', model, phase, '| Log:', log_path, flush=True)
                status = display('Đang nạp/kiểm model và checkpoint...', display_id=True)
                while process.poll() is None:
                    time.sleep(POLL_SECONDS)
                    progress = work / 'checkpoints' / model.replace('/', '_') / phase / 'progress.json'
                    if progress.exists():
                        status.update(model + ' | ' + phase + ' | ' + progress.read_text())
                    if time.time() - last_backup >= BACKUP_EVERY_SECONDS:
                        package_checkpoint()
                        last_backup = time.time()
                    if time.time() >= deadline + 60:
                        stop_process(process)
                        print('TẠM DỪNG: tiến trình được dừng, kết quả từng câu đã lưu.')
                        return False
                if process.returncode == 75:
                    print('TẠM DỪNG: worker đã lưu checkpoint.')
                    return False
                if process.returncode != 0:
                    print(''.join(log_path.read_text(errors='replace').splitlines(True)[-35:]))
                    raise RuntimeError(f'{model} lỗi; không chuyển sang full. Xem log và checkpoint ZIP.')
                print(''.join(log_path.read_text(errors='replace').splitlines(True)[-3:]))
        finally:
            if process is not None:
                stop_process(process)
            package_checkpoint()
    return True

## 4. Pilot 300 câu cho GPT-Neo 1.3B
PASS yêu cầu đúng ID/thứ tự, dựng lại prompt khớp từng chữ, nhãn chấm đúng và confidence hợp lệ. Không có ngưỡng accuracy giả định.

In [6]:
pilot_ok = run_phase(300)
print('PILOT PASS cho GPT-Neo 1.3B.' if pilot_ok else 'Pilot chưa đủ; tiếp tục ở phiên sau.')

Bắt đầu: EleutherAI/gpt-neo-1.3B check300 | Log: /kaggle/working/popqa-genread-neo13b/logs/EleutherAI_gpt-neo-1.3B-check300.log


'EleutherAI/gpt-neo-1.3B | check300 | {"done": 300, "total": 300, "seconds_per_new_answer": 6.168020177283333}'

290/300 | 6.17s/câu mới
300/300 | 6.17s/câu mới
PASS {"model": "EleutherAI/gpt-neo-1.3B", "method": "genread", "n": 300, "accuracy": 0.14, "has_answer": 0.09666666666666666, "empty_predictions": 0, "empty_passages": 0, "truncated_prompts": 0, "max_prompt_tokens": 1502, "ids_and_prompts_ok": true}

PILOT PASS cho GPT-Neo 1.3B.


## 5. Full 14.267 câu: GPT-Neo 1.3B
Chỉ chạy khi pilot GPT-Neo 1.3B PASS. Có thể cần nhiều phiên; thời gian thực được ghi trong log. Không tự hạ dtype hay giảm tập câu.

In [7]:
full_ok = False
if RUN_FULL and pilot_ok:
    full_ok = run_phase(0)
    print('FULL HOÀN TẤT cho GPT-Neo 1.3B.' if full_ok else 'Full chưa đủ; tiếp tục từ checkpoint ở phiên sau.')
elif not RUN_FULL:
    print('RUN_FULL=False: chỉ chạy pilot.')
else:
    print('Không chạy full vì pilot chưa hoàn tất.')

Bắt đầu: EleutherAI/gpt-neo-1.3B full | Log: /kaggle/working/popqa-genread-neo13b/logs/EleutherAI_gpt-neo-1.3B-full.log


'EleutherAI/gpt-neo-1.3B | full | {"done": 5120, "total": 14267, "seconds_per_new_answer": 6.191960581196094}'

KeyboardInterrupt: 

## 6. Bảng kết quả và file tải về
Chỉ CSV đã hoàn tất mới được xuất. File slim dùng được với `scripts/aggregate.py` của repo. Luôn giữ ZIP raw để phân tích passage/prompt.

In [ ]:
summaries = []
slim = work / 'results/slim'
slim.mkdir(exist_ok=True)
for p in sorted((work / 'results').glob('model=*.csv')):
    frame = pd.read_csv(p, keep_default_na=False)
    flags = frame.is_correct.astype(str).str.lower()
    assert flags.isin(['true','false']).all() and frame.id.is_unique
    n = 300 if 'check300' in p.name else 14267
    assert len(frame) == n
    assert frame.id.astype(int).tolist() == questions.sample(n=n, random_state=633).id.astype(int).tolist()
    summaries.append(dict(file=p.name, n=n, accuracy=float(flags.eq('true').mean()),
                          has_answer=float(frame.has_answer.astype(str).str.lower().eq('true').mean())))
    frame[['id','prop','s_pop','is_correct','has_answer','conf']].to_csv(slim / p.name, index=False)
if summaries:
    summary = pd.DataFrame(summaries)
    display(summary)
    summary.to_csv(work / 'results/summary-genread.csv', index=False)
else:
    print('Chưa có CSV hoàn chỉnh; checkpoint vẫn tải được.')
checkpoint_zip = package_checkpoint()
result_zip = Path('/kaggle/working/popqa-genread-neo13b-results.zip')
with zipfile.ZipFile(result_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    for folder in ['results','logs','data']:
        for p in (work / folder).rglob('*'):
            if p.is_file() and not p.name.endswith('.tmp'):
                z.write(p, str(p.relative_to(work)))
    for p in [runner, work / 'run_config.json']:
        z.write(p, p.name)
with zipfile.ZipFile(result_zip) as z:
    assert z.testzip() is None
os.chdir('/kaggle/working')
display(FileLink(result_zip.name))
display(FileLink(checkpoint_zip.name))
print('results.zip: CSV raw + slim + log + code. checkpoint.zip: Add Input để chạy tiếp.')

## Đóng gói lại sau khi bấm Interrupt
Nếu cell chạy báo lỗi hoặc bị ngắt, có thể chạy riêng cell dưới để lấy checkpoint mới nhất. Kill cứng toàn kernel không thể chạy `finally`.

In [8]:
checkpoint_zip = package_checkpoint()
os.chdir('/kaggle/working')
display(FileLink(checkpoint_zip.name))

/kaggle/working/popqa-genread-neo13b-checkpoint.zip